In [ ]:
import pandas as pd
from pathlib import Path
import re
from striprtf.striprtf import rtf_to_text

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

In [ ]:
DATA_DIR = Path("./data/csew")

EARLIEST_WAVE = 11
LATEST_WAVE = 23

KIND = "vf"   #"nvf" or "vf"

WAVES = {
    11: "2011-12",
    12: "2012-13",
    13: "2013-14",
    14: "2014-15",
    15: "2015-16",
    16: "2016-17",
    17: "2017-18",
    18: "2018-19",
    19: "2019-20",
    22: "2022-23",
    23: "2023-24",
}

In [ ]:
# Cell 3 — helper functions
def build_wave_prefix(wave_code):
    end = str(wave_code + 1).zfill(2)
    return f"csew_apr{wave_code:02d}mar{end}"


def get_rtf_path(wave_code, kind="nvf"):
    wave_folder = WAVES[wave_code]
    prefix = build_wave_prefix(wave_code)

    return (
        DATA_DIR
        / wave_folder
        / "mrdoc"
        / "ukda_data_dictionaries"
        / f"{prefix}_{kind}_ukda_data_dictionary.rtf"
    )


def extract_vars_from_rtf(rtf_path):
    with open(rtf_path, encoding="cp1252") as f:
        rtf = f.read()

    text = rtf_to_text(rtf)

    vars_found = re.findall(r"Variable = (\S+)", text)

    return sorted(set(vars_found))

In [ ]:
available_rtfs = []

for wave_code in range(EARLIEST_WAVE, LATEST_WAVE + 1):
    if wave_code not in WAVES:
        continue

    path = get_rtf_path(wave_code, kind=KIND)

    available_rtfs.append({
        "wave_code": wave_code,
        "wave": WAVES[wave_code],
        "kind": KIND,
        "path": path,
        "exists": path.exists()
    })

df_rtf_check = pd.DataFrame(available_rtfs)
df_rtf_check

In [ ]:
wave_vars = {}

for_check = df_rtf_check.copy()

for row in for_check.itertuples(index=False):
    if not row.exists:
        print(f"Missing: {row.path}")
        continue

    print(f"Loading {row.wave} ({row.kind}): {row.path}")

    vars_in_wave = extract_vars_from_rtf(row.path)

    wave_vars[row.wave] = set(vars_in_wave)

print(f"\nLoaded {len(wave_vars)} waves")

In [ ]:
all_vars = sorted(set().union(*wave_vars.values()))

df_var_waves = pd.DataFrame({"var": all_vars})

for wave, vars_set in wave_vars.items():
    df_var_waves[wave] = df_var_waves["var"].isin(vars_set)

df_var_waves.head()

In [ ]:
# Load manually curated keep/exclude sheet
vf_cols = pd.read_excel(
    f_root / "data/csew/vars_match/csew_vf_data_dictionary.xlsx"
)

# Keep variables marked "exclude"
vf_cols_exclude = vf_cols.loc[
    vf_cols["exclude"].eq("exclude"),
    "var"
].tolist()

print(f"Variables to exclude: {len(vf_cols_exclude):,}")

In [ ]:
df_var_waves_filtered = df_var_waves[
    ~df_var_waves["var"].isin(vf_cols_exclude)
].copy()

In [ ]:
print(df_var_waves_filtered.shape)

df_var_waves_filtered.drop(columns="var").sum()

In [ ]:
wave_columns = [c for c in df_var_waves_filtered.columns if c != "var"]

df_var_waves_filtered["n_waves_present"] = df_var_waves_filtered[wave_columns].sum(axis=1)

df_var_waves_filtered.sort_values(
    ["n_waves_present", "var"],
    ascending=[False, True]
).head(6)

In [ ]:
# Create mapping from var -> priority
priority_map = (
    vf_cols
    .set_index("var")["priority"]
)

# Add priority column to df_var_waves_filtered
df_var_waves_filtered["priority"] = (
    df_var_waves_filtered["var"]
    .map(priority_map)
)

df_var_waves_filtered.head()

In [ ]:
df_var_waves_filtered.to_csv(f_root / "data/csew/vars_match/csew_vf_cols_comparison.csv", index=False)